In [31]:
import json
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, HDBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.preprocessing import StandardScaler

In [32]:
# 1. Carica tracks.json
with open('tracks_cleaned.json', encoding='utf-8') as f:
    tracks = json.load(f)
df_tracks = pd.DataFrame(tracks)

## Hierarcical

In [33]:
def get_linkage_matrix(model):
    # Create linkage matrix 
    
    # create the counts of samples under each node
    counts = np.zeros(model.children_.shape[0])
    n_samples = len(model.labels_)
    for i, merge in enumerate(model.children_):
        current_count = 0
        for child_idx in merge:
            if child_idx < n_samples:
                current_count += 1  # leaf node
            else:
                current_count += counts[child_idx - n_samples]
        counts[i] = current_count

    linkage_matrix = np.column_stack(
        [model.children_, model.distances_, counts]
    ).astype(float)

    return linkage_matrix

def plot_dendrogram(model, **kwargs):
    linkage_matrix = get_linkage_matrix(model)
    dendrogram(linkage_matrix, **kwargs)

In [34]:
df_train = df_tracks.drop(['id', 'id_artist', 'title', 'featured_artists', 'primary_artist',
       'language', 'album', 'swear_IT', 'swear_EN', 'swear_IT_words',
       'swear_EN_words', 'year', 'month', 'day', 'n_sentences', 'n_tokens',
       'char_per_tok', 'avg_token_per_clause', #'bpm', 'rolloff', 'flux', 'rms',
       #'flatness', 'spectral_complexity', 'pitch', 'loudness', 
       'album_name', 'album_release_date', 'album_type', 'disc_number', 'track_number',
       'duration_ms', 'explicit', 'popularity', 'id_album', 'lyrics',
       'streams@1month','season', 'year_yyyy', 'month_yyyymm', 'day_yyyymmdd' ], axis=1)
scaler = StandardScaler()
scaler.fit(df_train)
df_train_2 = scaler.transform(df_train)

ValueError: could not convert string to float: 'Melodic'

The commented attributes are the ones that have been left in order to perform the clustering.

In [ ]:
model_2 = AgglomerativeClustering(distance_threshold=10,
                                n_clusters=None, metric='euclidean', linkage='complete')
model_2 = model_2.fit(df_train_2)
plt.title("Hierarchical Clustering Dendrogram - Complete LINK")
plot_dendrogram(model_2, truncate_mode="lastp", color_threshold=10)
plt.xlabel("Number of points in node (or index of point if no parenthesis).")
plt.show()

Single and average link struggled to separate the songs, while complete link manages to do so better. But several clusters are very small, so not the best for categorizing the songs.

## HDBSCAN

In [ ]:
hdb = HDBSCAN(cluster_selection_epsilon=0.5, min_samples=5, 
              min_cluster_size=10, max_cluster_size=15,
              store_centers="centroid")
hdb.fit(df_train_2)
df_tracks['hdbscan_labels'] = hdb.labels_
sns.scatterplot(data=df_tracks, 
                x="pitch",
                y="spectral_complexity", 
                hue=hdb.labels_, 
                style=hdb.labels_, 
                palette="bright")

#plt.scatter(scaler.inverse_transform(hdb.centroids_)[:,0], scaler.inverse_transform(hdb.centroids_)[:,2], c='red', marker='*', s=200)
plt.show()

Tried several combinations of epsilon and min_samples, but the way the songs are distributed means that there is one big cluster and several very small clusters

In [ ]:
df_tracks = df_tracks.drop(['hdbscan_labels'], axis=1)

## K-Means

In [ ]:
%%time
sse_list = []
sil_list = []

for k in range(2, 60):
    kmeans = KMeans(init='k-means++', n_clusters=k, n_init=10, max_iter=100)
    kmeans.fit(df_train_2)
    sse_list.append(kmeans.inertia_)
    sil_list.append(silhouette_score(df_train_2, kmeans.labels_))

In [ ]:
fig, axs = plt.subplots(2) 

sns.lineplot(x=range(2,len(sse_list)+2), y=sse_list, marker='o', ax=axs[0])
axs[0].set(xlabel='k', ylabel='SSE')

sns.lineplot(x=range(2,len(sil_list)+2), y=sil_list, marker='o', ax=axs[1])
axs[1].set(xlabel='k', ylabel='Silhouette')

plt.tight_layout() # Adjust the padding between and around subplots

7 clusters seem to be among the best choice based on SSE and silhouette. 6 clusters would have also been good, but the clusters had fewer attributes which were higher or lower than the mean, making it less clear how to define the categories.

In [ ]:
n_clust = 7
kmeans = KMeans(n_clusters=n_clust, n_init=100, max_iter=100, random_state=42)
kmeans.fit(df_train_2)

In [ ]:
df_tracks['kmeans_labels'] = kmeans.labels_

fig = plt.figure(figsize=(13, 8))

sns.countplot(data=df_tracks, x='kmeans_labels', hue='primary_artist', legend=False)
plt.show()

Cluster 0: high pitch -> Melodic

Cluster 1: lowest bpm, low rolloff -> Slow Dark

Cluster 2: lowest flatness -> Clean

Cluster 3: lowest rolloff, highest loudness, highest pitch, highest rms -> Warm Bangers

Cluster 4: highest rolloff, high loudness, highest spectral_complexity -> Hype

Cluster 5: low rolloff, lowest loudness, lowest spectral_complexity, lowest rms, lowest flux -> Minimal

Cluster 6: highest bpm, high spectral_complexity -> Fast-Flow

In [ ]:
label_to_cat = {0:'Melodic', 1:'Slow Dark', 2:'Clean', 3:'Warm Bangers', 4:'Hype', 5:'Minimal', 6:'Fast-Flow'}

In [ ]:
for i in range(n_clust):
    df = df_tracks.loc[df_tracks['kmeans_labels']==i, 'category'] = label_to_cat[i]

In [ ]:
df_tracks = df_tracks.drop(['kmeans_labels'], axis=1)
df_tracks

In [ ]:
# Convert DataFrame to list of dictionaries
df_tracks_json = df_tracks.to_dict(orient='records')

# Save using your exact formatting style
with open('tracks_cleaned.json', 'w', encoding='utf-8') as f:
    json.dump(df_tracks_json, f, indent=2, ensure_ascii=False)